# unbox-args-tensor-to-array — worked example 1: Unbox positional MiniTensor args

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `unbox-args-tensor-to-array`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Inside `wrap_forward_fn`, every `MiniTensor` argument must be replaced by its raw `.array` before the underlying numerical function runs, while non-Tensor arguments (ints, floats, strings) pass through unchanged. The gate is `isinstance(a, MiniTensor)` — never duck-typing on `.array`, since unrelated objects can also expose that attribute.

## Worked solution

We unbox a tuple of positional arguments so the raw function never sees a `MiniTensor`.

1. We iterate the args with a comprehension. For each `a`, the test `isinstance(a, MiniTensor)` decides whether to unbox.
2. If it is a `MiniTensor`, we substitute `a.array` — the same underlying object, not a copy, preserving identity for later `is`-checks.
3. Otherwise we keep `a` as-is, so scalars and strings survive untouched.
4. We return a new tuple in the original order.

We print the unboxed tuple to show tensors became their `.array` payloads while the float passed through.

In [ ]:
class MiniTensor:
    def __init__(self, array):
        self.array = array

def unbox_args(args: tuple) -> tuple:
    return tuple(a.array if isinstance(a, MiniTensor) else a for a in args)

t1 = MiniTensor([1, 2, 3])
t2 = MiniTensor([4, 5])
out = unbox_args((t1, 3.0, t2))
print('unboxed:', out)
print('t1 identity preserved:', out[0] is t1.array)
print('scalar passthrough:', out[1] == 3.0)